# 05 — Integración micro ESS–Eurostat

> **Pipeline:** 01 Auditoría → 02 Limpieza ESS → 03 Eurostat → 04 Construcción EPBI → **05 Integración micro–macro** → 06 Econometría → 07 Datos para dashboard

Este notebook es el **punto de convergencia** de las dos ramas preparadas previamente:

- el notebook 04 aporta los microdatos ESS con el EPBI individual;
- el notebook 03 aporta el panel macroeconómico Eurostat por país y año.

La integración añade el contexto macroeconómico a cada entrevistado **sin cambiar la unidad de análisis**, que continúa siendo el individuo ESS. La salida resultante será la base común para la econometría del notebook 06 y para la preparación descriptiva del dashboard en el notebook 07.

## Criterios de integración

1. Se conservan todos los individuos procedentes del notebook 04, incluidos aquellos con `epbi = NA`.
2. Para cada indicador macro se busca primero una coincidencia exacta por **país + año ESS**.
3. Si no existe dato exacto, se utiliza el valor disponible más cercano dentro de **±1 año**.
4. El emparejamiento temporal se realiza indicador por indicador.
5. En caso de empate entre el año anterior y el posterior, se prioriza el año anterior.
6. Para cada indicador se conserva el año realmente utilizado y su diferencia respecto al año ESS.
7. Las variables macro se repiten entre los individuos del mismo contexto porque son características país-año, no variables individuales.
8. `analysis_weight`, `psu` y `stratum` se preservan sin cambios.
9. La única salida es `epbi_micro_macro_panel.parquet`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

PROCESADOS = PROJECT_ROOT / "DATOS" / "PROCESADOS"
PROCESADOS.mkdir(parents=True, exist_ok=True)

ESS_INPUT = PROCESADOS / "ess_epbi_micro.parquet"
EUROSTAT_INPUT = PROCESADOS / "eurostat_macro_panel.parquet"

OUT_MICRO = PROCESADOS / "epbi_micro_macro_panel.parquet"

MAX_YEAR_DISTANCE = 1

MACRO_VARS = [
    "renta_media_eur",
    "ratio_quintiles_renta",
    "gini",
    "gini_antes_transferencias",
    "gini_pensiones_transferencias",
    "riesgo_pobreza",
    "brecha_pobreza",
    "sobrecarga_vivienda",
    "privacion_material_social_severa",
    "ratio_renta_mayores",
]

print("Entrada micro:", ESS_INPUT)
print("Entrada Eurostat:", EUROSTAT_INPUT)
print("Salida micro-macro:", OUT_MICRO)

Entrada micro: C:\Users\Alero\Desktop\PROYECTO_FINAL\DATOS\PROCESADOS\ess_epbi_micro.parquet
Entrada Eurostat: C:\Users\Alero\Desktop\PROYECTO_FINAL\DATOS\PROCESADOS\eurostat_macro_panel.parquet
Salida micro-macro: C:\Users\Alero\Desktop\PROYECTO_FINAL\DATOS\PROCESADOS\epbi_micro_macro_panel.parquet


NameError: name 'OUT_COUNTRY_ROUND' is not defined

## 1. Carga y validación de las dos ramas

La integración comienza cargando `ess_epbi_micro.parquet` del notebook 04 y `eurostat_macro_panel.parquet` del notebook 03. Antes del cruce se comprueba la unicidad de los individuos ESS y de la clave `cntry + year` en Eurostat, porque ambas condiciones son necesarias para evitar duplicaciones.


In [ ]:
if not ESS_INPUT.exists():
    raise FileNotFoundError(
        "No se encuentra ess_epbi_micro.parquet. Ejecuta primero el notebook 04."
    )

if not EUROSTAT_INPUT.exists():
    raise FileNotFoundError(
        "No se encuentra eurostat_macro_panel.parquet. Ejecuta primero el notebook 03."
    )

ess = pd.read_parquet(ESS_INPUT)
eu = pd.read_parquet(EUROSTAT_INPUT)

required_micro = [
    "respondent_id",
    "cntry",
    "essround",
    "survey_year",
    "epbi",
    "epbi_abs",
    "valid_epbi",
    "analysis_weight",
]

required_macro = ["cntry", "year"] + MACRO_VARS

missing_micro = [c for c in required_micro if c not in ess.columns]
missing_macro = [c for c in required_macro if c not in eu.columns]

if missing_micro:
    raise KeyError("Faltan variables micro: " + ", ".join(missing_micro))

if missing_macro:
    raise KeyError("Faltan variables Eurostat: " + ", ".join(missing_macro))

if ess["respondent_id"].duplicated().any():
    raise ValueError("respondent_id no es único en el fichero del notebook 04.")

if eu.duplicated(["cntry", "year"]).any():
    raise ValueError("Eurostat contiene duplicados país-año. Revisar notebook 03.")

ess["cntry"] = ess["cntry"].astype("string").str.upper().str.strip()
eu["cntry"] = eu["cntry"].astype("string").str.upper().str.strip()

ess["survey_year"] = pd.to_numeric(ess["survey_year"], errors="coerce").astype("Int64")
eu["year"] = pd.to_numeric(eu["year"], errors="coerce").astype("Int64")

print("Micro:", ess.shape)
print("Eurostat:", eu.shape)
print("Países ESS:", ess["cntry"].nunique())
print("Periodo ESS:", ess["survey_year"].min(), "-", ess["survey_year"].max())

## 2. Asignación de variables macro por país y año

El emparejamiento se calcula sobre las combinaciones únicas `cntry + survey_year`, no fila por fila. Así, todos los entrevistados de un mismo contexto reciben exactamente la misma información macroeconómica.

Para cada indicador se sigue esta secuencia:

1. utilizar el año exacto, si existe;
2. si falta, buscar el año más cercano dentro de ±1;
3. si hay empate, priorizar el año anterior;
4. si no existe información dentro de ese margen, mantener `NA`.

Esta regla conserva la máxima proximidad temporal posible sin utilizar valores demasiado alejados del momento de la encuesta.


In [ ]:
cells_ess = (
    ess[["cntry", "survey_year"]]
    .drop_duplicates()
    .sort_values(["cntry", "survey_year"])
    .reset_index(drop=True)
)

def nearest_macro_value(country, survey_year, variable):
    candidates = eu.loc[
        eu["cntry"].eq(country) & eu[variable].notna(),
        ["year", variable]
    ].copy()

    if candidates.empty or pd.isna(survey_year):
        return np.nan, pd.NA, pd.NA, "missing"

    candidates["signed_distance"] = (
        pd.to_numeric(candidates["year"], errors="coerce")
        - int(survey_year)
    )
    candidates["abs_distance"] = candidates["signed_distance"].abs()

    candidates = candidates.loc[
        candidates["abs_distance"] <= MAX_YEAR_DISTANCE
    ].copy()

    if candidates.empty:
        return np.nan, pd.NA, pd.NA, "missing"

    # Prioridad: menor distancia absoluta; en empate, año anterior.
    candidates["future_priority"] = (candidates["signed_distance"] > 0).astype(int)

    best = candidates.sort_values(
        ["abs_distance", "future_priority", "year"]
    ).iloc[0]

    used_year = int(best["year"])
    signed_distance = used_year - int(survey_year)
    match_type = "exact" if signed_distance == 0 else "nearest"

    return best[variable], used_year, signed_distance, match_type


macro_cell = cells_ess.copy()

for variable in MACRO_VARS:
    assigned = [
        nearest_macro_value(row.cntry, row.survey_year, variable)
        for row in macro_cell[["cntry", "survey_year"]].itertuples(index=False)
    ]

    assigned_df = pd.DataFrame(
        assigned,
        columns=[
            variable,
            f"{variable}_year_used",
            f"{variable}_year_diff",
            f"{variable}_match",
        ],
        index=macro_cell.index,
    )

    macro_cell = pd.concat([macro_cell, assigned_df], axis=1)

display(macro_cell.head())

## 3. Diagnóstico del emparejamiento temporal

Una vez asignados los indicadores, se resume cuántos contextos se han emparejado de forma exacta, aproximada o sin dato. También se registran los casos en los que se ha utilizado un año distinto al año ESS, de modo que la aproximación temporal quede completamente trazable.


In [ ]:
match_summary = []

for variable in MACRO_VARS:
    counts = (
        macro_cell[f"{variable}_match"]
        .value_counts(dropna=False)
        .reindex(["exact", "nearest", "missing"], fill_value=0)
    )

    match_summary.append({
        "variable": variable,
        "celdas_exactas": int(counts["exact"]),
        "celdas_nearest": int(counts["nearest"]),
        "celdas_sin_dato": int(counts["missing"]),
        "pct_exactas": round(100 * counts["exact"] / len(macro_cell), 2),
        "pct_con_dato_total": round(
            100 * (counts["exact"] + counts["nearest"]) / len(macro_cell), 2
        ),
    })

match_summary = pd.DataFrame(match_summary)
display(match_summary)

nearest_detail = []

for variable in MACRO_VARS:
    tmp = macro_cell.loc[
        macro_cell[f"{variable}_match"].eq("nearest"),
        [
            "cntry",
            "survey_year",
            f"{variable}_year_used",
            f"{variable}_year_diff",
        ],
    ].copy()

    if not tmp.empty:
        tmp.insert(0, "variable", variable)
        tmp = tmp.rename(columns={
            f"{variable}_year_used": "year_used",
            f"{variable}_year_diff": "year_diff",
        })
        nearest_detail.append(tmp)

if nearest_detail:
    nearest_detail = pd.concat(nearest_detail, ignore_index=True)
    display(nearest_detail.sort_values(["variable", "cntry", "survey_year"]))
else:
    nearest_detail = pd.DataFrame(
        columns=["variable", "cntry", "survey_year", "year_used", "year_diff"]
    )
    print("No ha sido necesario utilizar años cercanos.")

## 4. Merge micro + macro

Con el mapa país-año preparado, se realiza el cruce con los microdatos individuales. El `merge` debe preservar exactamente el número de entrevistados del notebook 04 y mantener `respondent_id` como identificador único.


In [ ]:
panel = ess.merge(
    macro_cell,
    on=["cntry", "survey_year"],
    how="left",
    validate="many_to_one",
)

if len(panel) != len(ess):
    raise RuntimeError(
        f"El merge ha alterado el número de individuos: "
        f"{len(ess):,} → {len(panel):,}"
    )

if panel["respondent_id"].duplicated().any():
    raise ValueError("El merge ha generado duplicados individuales.")

panel["macro_disponible"] = (
    panel[MACRO_VARS].notna().any(axis=1)
)

panel["macro_completo"] = (
    panel[MACRO_VARS].notna().all(axis=1)
)

print("Panel micro-macro:", panel.shape)
print("Individuos con al menos un indicador macro:", int(panel["macro_disponible"].sum()))
print("Individuos con todos los indicadores macro:", int(panel["macro_completo"].sum()))

## 5. Variables institucionales país-año

Después de incorporar Eurostat se añaden variables contextuales institucionales derivadas directamente de `cntry` y `survey_year`. Estas variables describen el contexto del año de la encuesta y, por tanto, no dependen del año Eurostat utilizado para un indicador concreto.


In [ ]:
# Estas variables describen el contexto institucional correspondiente al año ESS.
# Se calculan con survey_year, no con el año Eurostat utilizado para un indicador concreto.

def eu_member(country, year):
    if pd.isna(year):
        return pd.NA

    y = int(year)

    accession = {
        "AT": 1995, "BE": 1958, "BG": 2007, "CY": 2004, "CZ": 2004,
        "DE": 1958, "DK": 1973, "EE": 2004, "ES": 1986, "FI": 1995,
        "FR": 1958, "GR": 1981, "HR": 2013, "HU": 2004, "IE": 1973,
        "IT": 1958, "LT": 2004, "LU": 1958, "LV": 2004, "MT": 2004,
        "NL": 1958, "PL": 2004, "PT": 1986, "RO": 2007, "SE": 1995,
        "SI": 2004, "SK": 2004,
    }

    if country == "GB":
        return int(y < 2020)

    start = accession.get(country)
    return int(start is not None and y >= start)


def euro_currency(country, year):
    if pd.isna(year):
        return pd.NA

    y = int(year)

    euro_start = {
        "AT": 1999, "BE": 1999, "CY": 2008, "DE": 1999, "EE": 2011,
        "ES": 1999, "FI": 1999, "FR": 1999, "GR": 2001, "HR": 2023,
        "IE": 1999, "IT": 1999, "LT": 2015, "LU": 1999, "LV": 2014,
        "NL": 1999, "PT": 1999, "SI": 2007, "SK": 2009,
    }

    start = euro_start.get(country)
    return int(start is not None and y >= start)


panel["eu_member"] = [
    eu_member(c, y) for c, y in zip(panel["cntry"], panel["survey_year"])
]

panel["euro_currency"] = [
    euro_currency(c, y) for c, y in zip(panel["cntry"], panel["survey_year"])
]

# Israel es el único país ESS del diseño actual que se marca fuera de Europa.
panel["europe"] = (~panel["cntry"].eq("IL")).astype("int8")

panel["eu_member"] = pd.array(panel["eu_member"], dtype="Int8")
panel["euro_currency"] = pd.array(panel["euro_currency"], dtype="Int8")

display(
    panel[
        ["cntry", "survey_year", "eu_member", "euro_currency", "europe"]
    ]
    .drop_duplicates()
    .sort_values(["cntry", "survey_year"])
    .head(30)
)

## 6. Controles finales

Antes de guardar el panel se verifica que ningún indicador macro se haya asignado a más de ±1 año, que el merge no haya creado duplicados y que tanto el EPBI como `analysis_weight` se hayan conservado sin alteraciones respecto al notebook 04.


In [ ]:
# Ningún año macro puede estar a más de ±1 año.
year_diff_cols = [f"{v}_year_diff" for v in MACRO_VARS]

distance_ok = True
for col in year_diff_cols:
    vals = pd.to_numeric(panel[col], errors="coerce").dropna()
    if not vals.abs().le(MAX_YEAR_DISTANCE).all():
        distance_ok = False
        break

checks = pd.DataFrame([
    {
        "control": "Una fila por respondent_id",
        "resultado": "OK" if not panel["respondent_id"].duplicated().any() else "REVISAR",
    },
    {
        "control": "Mismo número de individuos que notebook 04",
        "resultado": "OK" if len(panel) == len(ess) else "REVISAR",
    },
    {
        "control": "Año macro máximo ±1",
        "resultado": "OK" if distance_ok else "REVISAR",
    },
    {
        "control": "EPBI individual sin modificar",
        "resultado": "OK" if panel["epbi"].equals(ess["epbi"]) else "REVISAR",
    },
    {
        "control": "analysis_weight conservado",
        "resultado": "OK" if panel["analysis_weight"].equals(ess["analysis_weight"]) else "REVISAR",
    },
])

display(checks)

if (checks["resultado"] != "OK").any():
    raise ValueError("Alguno de los controles finales del notebook 05 requiere revisión.")

print("Individuos:", len(panel))
print("EPBI válidos:", int(panel["valid_epbi"].fillna(False).sum()))
print("Individuos con al menos un indicador macro:", int(panel["macro_disponible"].sum()))
print("Individuos con todos los indicadores macro:", int(panel["macro_completo"].sum()))

## 7. Exportación

El resultado se guarda en un único Parquet individual:

`DATOS/PROCESADOS/epbi_micro_macro_panel.parquet`

Cada fila representa un entrevistado ESS. Los indicadores macro aparecen repetidos entre los individuos del mismo país-año porque describen un **contexto compartido**.

Esta repetición es deliberada: en análisis individuales permite combinar características personales y contextuales. En agregaciones y en el dashboard, los macrodatos no deben sumarse ni tratarse como si fueran observaciones independientes.

No se generan CSV, Excel, Pickle ni tablas agregadas en esta fase.


In [ ]:
panel.to_parquet(OUT_MICRO, index=False)

print("Archivo guardado:")
print(" -", OUT_MICRO)
print("Filas:", len(panel))
print("Columnas:", panel.shape[1])

## 8. Resultado de la fase 05 y continuidad

La salida `epbi_micro_macro_panel.parquet` reúne por primera vez toda la información analítica del proyecto en una sola estructura individual:

- EPBI y variables derivadas;
- características sociodemográficas y actitudinales ESS;
- `analysis_weight`, `psu` y `stratum`;
- indicadores macroeconómicos Eurostat;
- año macro utilizado para cada indicador;
- diferencia temporal entre el año macro y el año ESS;
- variables institucionales (`eu_member`, `euro_currency`, `europe`).

Los individuos con `epbi = NA` se conservan por trazabilidad, aunque no entrarán en modelos que requieran un EPBI válido.

A partir de aquí el pipeline deja de preparar datos y pasa al análisis. El **notebook 06** utiliza este panel para la econometría. Después, el **notebook 07** combinará este mismo panel con los resultados del 06 para construir las dos fuentes de datos del dashboard.
